In [5]:
import pandas as pd
import os
import glob
import re

csv_files = glob.glob("lsq_data/*.csv")


all_queries = set()
combined_df = pd.DataFrame(columns=["text"])


total_queries = 0
non_select_queries = 0
duplicate_queries = 0
temp = []
for file in csv_files:
    df = pd.read_csv(file)
    if "text" not in df.columns:
        continue

    for query in df["text"].dropna().astype(str):
        total_queries += 1


        normalized_query = re.sub(r"\s+", " ", query).strip().lower()

        #  Check if it's a SELECT query
        if any(kw in normalized_query for kw in ["ask", "construct", "describe"]):
            non_select_queries += 1
           
            continue
        #if not normalized_query.startswith("select"):
         #   non_select_queries += 1
          #  continue

        # Skip duplicates
        if normalized_query in all_queries:
            duplicate_queries += 1
            continue

        # Add to set and final DataFrame
        all_queries.add(normalized_query)
        combined_df = pd.concat([combined_df, pd.DataFrame({"text": [query]})], ignore_index=True)


combined_df.to_csv("combined_unique_select_queries.csv", index=False)

# stats
print("=== Summary ===")
print(f"Total queries processed: {total_queries}")
print(f"Non-SELECT queries skipped: {non_select_queries}")
print(f"Duplicate SELECT queries skipped: {duplicate_queries}")
print(f"Unique SELECT queries saved: {len(combined_df)}")


=== Summary ===
Total queries processed: 320000
Non-SELECT queries skipped: 11566
Duplicate SELECT queries skipped: 881
Unique SELECT queries saved: 307553
